<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/NAT_DS_TUTORIAL_GPT_OSS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!uv pip install "nvidia-nat[all]"

In [2]:
from google.colab import userdata
api_key = userdata.get('NVIDIA_API_KEY')
import os
os.environ['NVIDIA_API_KEY'] = api_key

api_key_TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')
os.environ['TAVILY_API_KEY'] = api_key_TAVILY_API_KEY

## CASE 1: Basic ReAct Agent & Local Tools

In [ ]:
!pip install "nvidia-nat[langchain]"

In [18]:
!curl -s https://integrate.api.nvidia.com/v1/models | python -m json.tool | grep '"id"'

            "id": "01-ai/yi-large",
            "id": "adept/fuyu-8b",
            "id": "ai21labs/jamba-1.5-large-instruct",
            "id": "aisingapore/sea-lion-7b-instruct",
            "id": "bigcode/starcoder2-15b",
            "id": "databricks/dbrx-instruct",
            "id": "deepseek-ai/deepseek-coder-6.7b-instruct",
            "id": "deepseek-ai/deepseek-v4-flash-0731",
            "id": "deepseek-ai/deepseek-v4-pro-0813",
            "id": "google/codegemma-1.1-7b",
            "id": "google/codegemma-7b",
            "id": "google/deplot",
            "id": "google/diffusiongemma-26b-a4b-it",
            "id": "google/gemma-2b",
            "id": "google/gemma-3-12b-it",
            "id": "google/gemma-3-4b-it",
            "id": "google/gemma-4-31b-it",
            "id": "google/recurrentgemma-2b",
            "id": "ibm/granite-3.0-3b-a800m-instruct",
            "id": "ibm/granite-3.0-8b-instruct",
            "id": "ibm/granite-34b-code-instruct",
            "id":

In [30]:
# ============================================================
# ART-II MISSION — DIAGNOSTIC RUN
# ============================================================
from google.colab import userdata
import os
import yaml
import subprocess

api_key = userdata.get('NVIDIA_API_KEY')
os.environ['NVIDIA_API_KEY'] = api_key
api_key_TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')
os.environ['TAVILY_API_KEY'] = api_key_TAVILY_API_KEY

config_dict = {
    "llms": {
        "reasoning_llm": {
            "_type": "openai",
            "model_name": "openai/gpt-oss-20b",
            "base_url": "https://integrate.api.nvidia.com/v1",
            "api_key": os.environ['NVIDIA_API_KEY'],
            "max_tokens": 4096,
            "temperature": 0.2,
        }
    },
    "functions": {
        "current_time": {"_type": "nat.tool/current_datetime"}
    },
    "workflow": {
        "_type": "tool_calling_agent",
        "llm_name": "reasoning_llm",
        "tool_names": ["current_time"],
        "verbose": True,
        "handle_tool_errors": True,
    },
}

with open("stable_workflow.yml", "w") as f:
    yaml.dump(config_dict, f, default_flow_style=False)

mission_input = (
    "Analyze ART-II telemetry (Power: 142W, CO2: 5.2 ppm) "
    "and provide a diagnosis with a timestamp. "
    "You MUST call the current_time tool to obtain the timestamp."
)

print("🚀 STARTING MISSION - DIAGNOSTIC")
print("-" * 70)

child_env = os.environ.copy()
child_env.update({
    "PYTHONWARNINGS": "ignore",
    "PYTHONNOUSERSITE": "1",
    "LANGCHAIN_OPENAI_TCP_KEEPALIVE": "0",
    "LANGCHAIN_OPENAI_STREAM_CHUNK_TIMEOUT_S": "0",
    "DATASETS_VERBOSITY": "error",
    "TRANSFORMERS_VERBOSITY": "error",
    "NAT_LOG_LEVEL": "ERROR",
})

# Run WITHOUT capture — let output stream directly to the notebook
result = subprocess.run(
    ["nat", "run", "--config_file", "stable_workflow.yml", "--input", mission_input],
    env=child_env,
    text=True,
)

print("-" * 70)
print(f"Return code: {result.returncode}")

🚀 STARTING MISSION - DIAGNOSTIC
----------------------------------------------------------------------
----------------------------------------------------------------------
Return code: 0


In [31]:
!nat run --config_file stable_workflow.yml --input "Your input" 2> >(grep -v -E "Authlib|authlib|compatible before" >&2)

2026-09-12 23:44:22 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'stable_workflow.yml'

Configuration Summary:
--------------------
Workflow Type: tool_calling_agent
Number of Functions: 1
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2026-09-12 23:44:23 - INFO     - nat.runtime.session:332 - Shared workflow built (entry_function=None)
2026-09-12 23:44:23 - INFO     - httpx:1740 - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-12 23:44:28 - INFO     - nat.plugins.langchain.agent.tool_calling_agent.agent:152 - 
------------------------------
[AGENT]
Agent input: Your input
Agent's thoughts: 
I’m happy to help, but I’m not sure what you’d like me to give input on. Could you let me know the topic or question you have in mind?
--------------------------

In [33]:
# ============================================================
# ART-II MISSION — SINGLE CELL WORKING SOLUTION
# ============================================================
from google.colab import userdata
import os
import yaml
import sys

# ------------------------------------------------------------
# 1. SETUP ENVIRONMENT
# ------------------------------------------------------------
api_key = userdata.get('NVIDIA_API_KEY')
os.environ['NVIDIA_API_KEY'] = api_key

api_key_TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')
os.environ['TAVILY_API_KEY'] = api_key_TAVILY_API_KEY

# ------------------------------------------------------------
# 2. WORKFLOW CONFIGURATION
# ------------------------------------------------------------
config_dict = {
    "llms": {
        "reasoning_llm": {
            "_type": "openai",
            "model_name": "openai/gpt-oss-20b",
            "base_url": "https://integrate.api.nvidia.com/v1",
            "api_key": os.environ['NVIDIA_API_KEY'],
            "max_tokens": 4096,
            "temperature": 0.2,
        }
    },
    "functions": {
        "current_time": {
            "_type": "nat.tool/current_datetime"
        }
    },
    "workflow": {
        "_type": "tool_calling_agent",
        "llm_name": "reasoning_llm",
        "tool_names": ["current_time"],
        "verbose": True,
        "handle_tool_errors": True,
    },
}

with open("stable_workflow.yml", "w") as f:
    yaml.dump(config_dict, f, default_flow_style=False)

# ------------------------------------------------------------
# 3. EXECUTION (bash + stderr filter — suppresses Authlib warning)
# ------------------------------------------------------------
mission_input = (
    "Analyze ART-II telemetry (Power: 142W, CO2: 5.2 ppm) "
    "and provide a diagnosis with a timestamp. "
    "You MUST call the current_time tool to obtain the timestamp "
    "before producing your final diagnosis."
)

print("🚀 STARTING MISSION")
print("-" * 70)

!bash -c 'nat run --config_file stable_workflow.yml --input "{mission_input}" 2> >(grep -vE "Authlib|authlib|compatible before" >&2)'

🚀 STARTING MISSION
----------------------------------------------------------------------
2026-09-12 23:47:18 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'stable_workflow.yml'

Configuration Summary:
--------------------
Workflow Type: tool_calling_agent
Number of Functions: 1
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2026-09-12 23:47:19 - INFO     - nat.runtime.session:332 - Shared workflow built (entry_function=None)
2026-09-12 23:47:19 - INFO     - httpx:1740 - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-12 23:47:20 - INFO     - nat.plugins.langchain.agent.tool_calling_agent.agent:152 - 
------------------------------
[AGENT]
Agent input: Analyze ART-II telemetry (Power: 142W, CO2: 5.2 ppm) and provide a diagnosis with a timestamp. You MU

## CASE 2: Template Validation & Literal Escaping

In [35]:
!pip install -q nemo-agent-toolkit-tavily

In [36]:
config_dict = {
    "llms": {
        "reasoning_llm": {
            "_type": "openai",
            "model_name": "openai/gpt-oss-20b",
            "base_url": "https://integrate.api.nvidia.com/v1",
            "api_key": os.environ['NVIDIA_API_KEY'],
            "max_tokens": 4096,
            "temperature": 0.2,
        }
    },
    "functions": {
        "current_time": {
            "_type": "nat.tool/current_datetime"
        },
    },
    "function_groups": {
        "internet_search": {
            "_type": "tavily",
            "include": ["search"],
        },
    },
    "workflow": {
        "_type": "tool_calling_agent",
        "llm_name": "reasoning_llm",
        "tool_names": ["current_time", "internet_search__search"],  # note the double underscore
        "verbose": True,
        "handle_tool_errors": True,
    },
}

In [37]:
# ============================================================================
# ART-II MISSION DIAGNOSTICS — CORRECTED ONE-CELL SOLUTION
# ============================================================================
from google.colab import userdata
import os
import yaml

# 1. SETUP ENVIRONMENT
os.environ['NVIDIA_API_KEY'] = userdata.get('NVIDIA_API_KEY')
os.environ['TAVILY_API_KEY'] = userdata.get('TAVILY_API_KEY')

# 2. WORKFLOW CONFIGURATION
config_dict = {
    "llms": {
        "reasoning_llm": {
            "_type": "openai",
            "model_name": "openai/gpt-oss-20b",
            "base_url": "https://integrate.api.nvidia.com/v1",
            "api_key": os.environ['NVIDIA_API_KEY'],
            "max_tokens": 4096,
            "temperature": 0.2,
        }
    },
    "functions": {
        "current_time": {
            "_type": "nat.tool/current_datetime"
        },
    },
    "function_groups": {
        "internet_search": {
            "_type": "tavily",
            "include": ["search"],
        },
    },
    "workflow": {
        "_type": "tool_calling_agent",
        "llm_name": "reasoning_llm",
        "tool_names": ["current_time", "internet_search__search"],
        "verbose": True,
        "handle_tool_errors": True,
    },
}

with open("stable_workflow.yml", "w") as f:
    yaml.dump(config_dict, f, default_flow_style=False)

# 3. EXECUTION
mission_input = (
    "1. Get the current timestamp. "
    "2. Diagnose 142W / 5.2 ppm telemetry based on the provided NASA standards. "
    "If you need to verify NASA ECLSS CO2 or scrubber power limits, use the web search tool."
)

print("🚀 STARTING MISSION")
print("-" * 70)

!bash -c 'nat run --config_file stable_workflow.yml --input "{mission_input}" 2> >(grep -vE "Authlib|authlib|compatible before" >&2)'

🚀 STARTING MISSION
----------------------------------------------------------------------
2026-09-12 23:50:25 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'stable_workflow.yml'

Configuration Summary:
--------------------
Workflow Type: tool_calling_agent
Number of Functions: 1
Number of Function Groups: 1
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2026-09-12 23:50:26 - INFO     - nat.runtime.session:332 - Shared workflow built (entry_function=None)
2026-09-12 23:50:26 - INFO     - httpx:1740 - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-12 23:50:29 - INFO     - nat.plugins.langchain.agent.tool_calling_agent.agent:152 - 
------------------------------
[AGENT]
Agent input: 1. Get the current timestamp. 2. Diagnose 142W / 5.2 ppm telemetry based on the provided NASA standar

## CASE 3:

In [41]:
# ============================================================================
# ART-II MISSION DIAGNOSTICS — STABLE ONE-CELL SOLUTION
# ============================================================================
from google.colab import userdata
import os
import yaml

# 1. SETUP ENVIRONMENT
os.environ['NVIDIA_API_KEY'] = userdata.get('NVIDIA_API_KEY')

# 2. WORKFLOW CONFIGURATION
config_dict = {
    "llms": {
        "reasoning_llm": {
            "_type": "openai",
            "model_name": "openai/gpt-oss-20b",
            "base_url": "https://integrate.api.nvidia.com/v1",
            "api_key": os.environ['NVIDIA_API_KEY'],
            "max_tokens": 4096,
            "temperature": 0.2,
        }
    },
    "functions": {
        "current_time": {"_type": "nat.tool/current_datetime"},
    },
    "workflow": {
        "_type": "tool_calling_agent",
        "llm_name": "reasoning_llm",
        "tool_names": ["current_time"],
        "verbose": True,
        "handle_tool_errors": True,
        "max_iterations": 4,   # hard cap — one tool call is enough
        "system_prompt": (
            "You are an ART-II mission diagnostics agent. "
            "Analyze the telemetry provided in the user's message. "
            "Call the current_time tool ONCE to obtain a timestamp. "
            "Then produce your final answer immediately — do NOT call any other tools. "
            "NASA ECLSS REFERENCE DATA: "
            "CO2 standard for crewed spacecraft is < 3900 ppm. "
            "Scrubber nominal power draw is around 120-150 W. "
            "Use this reference data to evaluate the telemetry."
        ),
    },
}

with open("stable_workflow.yml", "w") as f:
    yaml.dump(config_dict, f, default_flow_style=False)

# 3. EXECUTION
mission_input = (
    "Diagnose ART-II telemetry: Power: 142W, CO2: 5.2 ppm. "
    "Compare against NASA ECLSS standards and provide a timestamped diagnosis."
)

print("🚀 STARTING MISSION")
print("-" * 70)

!bash -c 'nat run --config_file stable_workflow.yml --input "{mission_input}" 2> >(grep -vE "Authlib|authlib|compatible before" >&2)'

🚀 STARTING MISSION
----------------------------------------------------------------------
2026-09-12 23:59:38 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'stable_workflow.yml'

Configuration Summary:
--------------------
Workflow Type: tool_calling_agent
Number of Functions: 1
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2026-09-12 23:59:39 - INFO     - nat.runtime.session:332 - Shared workflow built (entry_function=None)
2026-09-12 23:59:39 - INFO     - httpx:1740 - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-12 23:59:41 - INFO     - nat.plugins.langchain.agent.tool_calling_agent.agent:152 - 
------------------------------
[AGENT]
Agent input: Diagnose ART-II telemetry: Power: 142W, CO2: 5.2 ppm. Compare against NASA ECLSS standards and provid

## CASE 4: Stable Multi-Tool Configuration

In [43]:
# ============================================================================
# ART-II MISSION — FINAL STABLE ONE-CELL SOLUTION
# ============================================================================
from google.colab import userdata
import os
import yaml

os.environ['NVIDIA_API_KEY'] = userdata.get('NVIDIA_API_KEY')
os.environ['TAVILY_API_KEY'] = userdata.get('TAVILY_API_KEY')

config_dict = {
    "llms": {
        "reasoning_llm": {
            "_type": "openai",
            "model_name": "openai/gpt-oss-20b",
            "base_url": "https://integrate.api.nvidia.com/v1",
            "api_key": os.environ['NVIDIA_API_KEY'],
            "max_tokens": 4096,
            "temperature": 0.2,
        }
    },
    "functions": {
        "current_time": {"_type": "nat.tool/current_datetime"},
    },
    "function_groups": {
        "internet_search": {
            "_type": "tavily",
            "include": ["search"],
        },
    },
    "workflow": {
        "_type": "tool_calling_agent",
        "llm_name": "reasoning_llm",
        "tool_names": ["current_time", "internet_search__search"],
        "verbose": True,
        "handle_tool_errors": True,
        "max_iterations": 5,
        "system_prompt": (
            "You are an ART-II mission diagnostics agent. "
            "Analyze the telemetry provided in the user's message. "
            "1) Call current_time ONCE to obtain a timestamp. "
            "2) Optionally call internet_search__search AT MOST TWICE to verify NASA ECLSS reference values "
            "(use search_depth='basic'). "
            "3) Then produce the final answer IMMEDIATELY — do NOT keep searching. "
            "NASA ECLSS REFERENCE DATA: "
            "CO2 standard for crewed spacecraft is < 3900 ppm. "
            "Scrubber nominal power draw is around 120-150 W."
        ),
    },
}

with open("stable_workflow.yml", "w") as f:
    yaml.dump(config_dict, f, default_flow_style=False)

mission_input = (
    "Analyze ART-II telemetry: Power: 142W, CO2: 5.2 ppm. "
    "Provide a timestamped diagnosis compared against NASA ECLSS standards."
)

print("🚀 STARTING MISSION")
print("-" * 70)

!bash -c 'nat run --config_file stable_workflow.yml --input "{mission_input}" 2> >(grep -vE "Authlib|authlib|compatible before" >&2)'

🚀 STARTING MISSION
----------------------------------------------------------------------
2026-09-13 00:03:32 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'stable_workflow.yml'

Configuration Summary:
--------------------
Workflow Type: tool_calling_agent
Number of Functions: 1
Number of Function Groups: 1
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2026-09-13 00:03:34 - INFO     - nat.runtime.session:332 - Shared workflow built (entry_function=None)
2026-09-13 00:03:34 - INFO     - httpx:1740 - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-13 00:03:41 - INFO     - nat.plugins.langchain.agent.tool_calling_agent.agent:152 - 
------------------------------
[AGENT]
Agent input: Analyze ART-II telemetry: Power: 142W, CO2: 5.2 ppm. Provide a timestamped diagnosis compared against

## CASE 5: One-Shot Finalization & Code Execution

In [45]:
# ============================================================================
# ART-II MISSION — ONE-SHOT FINALIZATION (stable)
# ============================================================================
from google.colab import userdata
import os
import yaml

os.environ['NVIDIA_API_KEY'] = userdata.get('NVIDIA_API_KEY')
os.environ['TAVILY_API_KEY'] = userdata.get('TAVILY_API_KEY')

config_dict = {
    "llms": {
        "reasoning_llm": {
            "_type": "openai",
            "model_name": "openai/gpt-oss-20b",
            "base_url": "https://integrate.api.nvidia.com/v1",
            "api_key": os.environ['NVIDIA_API_KEY'],
            "max_tokens": 4096,
            "temperature": 0.2,
        }
    },
    "functions": {
        "current_time": {"_type": "nat.tool/current_datetime"},
        "python_engine": {
            "_type": "nat.plugins.langchain.tools/code_generation",
            "llm_name": "reasoning_llm",
        },
    },
    "function_groups": {
        "internet_search": {"_type": "tavily", "include": ["search"]},
    },
    "workflow": {
        "_type": "tool_calling_agent",
        "llm_name": "reasoning_llm",
        "tool_names": ["current_time", "python_engine", "internet_search__search"],
        "verbose": True,
        "handle_tool_errors": True,
        "max_iterations": 5,
        "system_prompt": (
            "You are a Lead Aerospace Engineer AI. "
            "Execute this plan in order and then STOP: "
            "1) Call current_time ONCE to log a timestamp. "
            "2) Optionally call internet_search__search AT MOST ONCE to look up NASA ECLSS CO2 benchmarks "
            "(use search_depth='basic'). "
            "3) Call python_engine ONCE to evaluate: (0.000001 * 50 * 86400 * 26000) / 3000. "
            "4) Immediately produce the Final Answer consolidating: "
            "the timestamp, the CO2 benchmark comparison, the numeric result, and a short ART-II diagnostic. "
            "Do NOT call any tool more than once. Do NOT keep searching."
        ),
    },
}

with open("final_mission.yml", "w") as f:
    yaml.dump(config_dict, f, default_flow_style=False)

print("🚀 STARTING ONE-SHOT MISSION")
print("-" * 70)

!bash -c 'nat run --config_file final_mission.yml --input "Conduct the ART-II diagnostic and calculate station-keeping for 50m." 2> >(grep -vE "Authlib|authlib|compatible before|LangChainDeprecationWarning|Calling .text\(\) as a method" >&2)'

🚀 STARTING ONE-SHOT MISSION
----------------------------------------------------------------------
2026-09-13 00:08:16 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'final_mission.yml'
2026-09-13 00:08:16 - INFO     - nat.plugins.langchain.tools.code_generation_tool:43 - Initializing code generation tool
Getting tool LLM from config
2026-09-13 00:08:17 - INFO     - nat.plugins.langchain.tools.code_generation_tool:54 - Filling tool's prompt variable from config
2026-09-13 00:08:17 - INFO     - nat.plugins.langchain.tools.code_generation_tool:57 - Initialized code generation tool

Configuration Summary:
--------------------
Workflow Type: tool_calling_agent
Number of Functions: 2
Number of Function Groups: 1
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2026-09-13 00:08:17 - INFO     - nat.runtime.session:332 - Shared workflow bui

# Summary — NAT Mission Notebook, Five Cases

A tutorial notebook that builds up a working NeMo Agent Toolkit (NAT) workflow in five progressively complex cases, all running against NVIDIA's NIM endpoint with the `openai/gpt-oss-20b` model.

---

## Setup Cells (run once)

1. `!uv pip install "nvidia-nat[all]"` — installs the framework
2. Load `NVIDIA_API_KEY` and `TAVILY_API_KEY` from Colab Secrets
3. `!pip install -q "nvidia-nat[langchain]"` — enables `tool_calling_agent` (required — the LangChain wrapper plugin is not installed by default)
4. `!curl …/v1/models` — optional sanity check of the models available to the account
5. `!pip install -q nemo-agent-toolkit-tavily` — installs the Tavily plugin (required because Tavily was moved out of `nvidia-nat[langchain]` in v1.8)

---

## CASE 1 — Basic ReAct agent with a local tool

**Purpose:** Establish the minimal working agent.

- **Model:** `openai/gpt-oss-20b`
- **Agent:** `tool_calling_agent`
- **Tools:** `current_time` (via `nat.tool/current_datetime`)
- **Input:** Analyse ART-II telemetry and timestamp the diagnosis

**Key learning:** The `tool_calling_agent` correctly calls `current_time` once, then produces the final answer. Confirms the LLM works and the basic ReAct loop functions with native function calling.

---

## CASE 2 — Add Tavily web search

**Purpose:** Demonstrate the function-group syntax for external tools.

- Adds a `function_groups` block with `_type: tavily`, `include: [search]`
- Tool referenced as `internet_search__search` (double-underscore syntax)
- **Input:** Diagnose telemetry, use web search to verify NASA ECLSS limits

**Key learning:** Tavily tools use a different config section than simple functions. The model also demonstrates self-correcting behaviour: it first calls the tool with `search_depth: 1` (numeric), gets a validation error, then retries with `"basic"`. The `handle_tool_errors: True` setting lets the agent recover automatically.

---

## CASE 3 — Constrained single-tool agent

**Purpose:** Eliminate failure modes by removing search and hard-capping iterations.

- Removes Tavily entirely — no function groups
- Adds `max_iterations: 4` (tight cap)
- Adds an explicit `system_prompt` with embedded NASA reference data:
  - CO₂ standard < 3900 ppm
  - Scrubber nominal power 120–150 W
- Prompt instructs: *"Call the current_time tool ONCE … then produce your final answer immediately — do NOT call any other tools"*

**Key learning:** Embedding reference data in the prompt lets the agent skip web search entirely. Result: 2 LLM calls, 1 tool call, ~7 seconds, deterministic output. This is the most reliable configuration for a task with known reference values.

---

## CASE 4 — Stable multi-tool with bounded search

**Purpose:** Re-enable Tavily but keep the agent from looping indefinitely.

- Restores `internet_search__search` alongside `current_time`
- `max_iterations: 5`
- Prompt explicitly bounds usage: *"call internet_search__search AT MOST TWICE … then produce the final answer IMMEDIATELY — do NOT keep searching"*

**Key learning:** Bounding the search rule is what makes multi-tool ReAct stable. Without a hard cap and a prompt-level "at most N" instruction, reasoning models like `gpt-oss-20b` will keep refining their query indefinitely.

---

## CASE 5 — One-shot finalization with code execution

**Purpose:** Full pipeline: timestamp → search → compute → consolidated report.

- Adds a third tool: `python_engine` via `nat.plugins.langchain.tools/code_generation`
- Prompt gives a 4-step plan: (1) `current_time` once, (2) optional search at most once, (3) `python_engine` once to evaluate `(0.000001 * 50 * 86400 * 26000) / 3000`, (4) produce final answer
- Warning filter extends to include `LangChainDeprecationWarning` and `Calling .text() as a method`

**Key learning:** Three tools can coexist cleanly as long as each is called exactly once. The agent executes all three in order and produces a structured report. Note the model sometimes fabricates physical interpretations for the arithmetic result (e.g., calling `37.44` "37.44 kg CO₂") — the prompt should supply units if physical meaning matters.

---

## Common Configuration Patterns

| Element | Working form | Note |
|---|---|---|
| LLM provider | `"_type": "openai"` | Not `"nim"` — invalid tag |
| Model field | `"model_name": "openai/gpt-oss-20b"` | Not `"model"` |
| Agent type | `"_type": "tool_calling_agent"` | Required for reasoning models; `react_agent` returns empty output |
| Simple tool | `functions: { name: {_type: nat.tool/...} }` | For single-purpose built-in tools |
| Provider tools | `function_groups: { group: {_type: tavily, include: [search]} }` | Referenced as `group__tool` |
| Error handling | `"handle_tool_errors": True` | Replaces `handle_parsing_errors` for tool-calling agents |
| Loop cap | `"max_iterations": 4–5` | Prevents unbounded tool calling |
| Warning suppression | `!bash -c 'nat run ... 2> >(grep -vE "Authlib\|..." >&2)'` | Uses bash process substitution; the only reliable filter |

---

## Common Failure Modes and Their Fixes

| Error | Cause | Fix |
|---|---|---|
| `no registered conversion to LLM framework: langchain` | LangChain plugin missing | `pip install "nvidia-nat[langchain]"` |
| `nat.agent/react_agent not found` | Wrong discriminator tag | Use `react_agent`, not `nat.agent/react_agent` |
| `404 page not found` (plain text) | Model not in account catalog | Switch to `openai/gpt-oss-20b` |
| `Function '...' not found for account` | Model in catalog, not provisioned | Same — pick a working model |
| `ReActAgentParsingFailedError: LLM output ''` | Reasoning model in ReAct mode | Switch to `tool_calling_agent` |
| `ToolInvocationError: search_depth should be ...` | Model passed numeric instead of string | Prompt hint: `search_depth='basic'` |
| `tavily_internet_search was removed ...` | Tavily relocated in v1.8 | `pip install nemo-agent-toolkit-tavily`, use function groups |
| Max iteration limit reached | Unbounded search loop | Tight `max_iterations` + explicit stop rule |
| Empty output from `subprocess.run` | NAT writes to TTY, not pipes | Use `!bash -c '...'` shell magic |

---

## Summary Table — Five Cases at a Glance

| Case | Agent | Tools | Iteration cap | System prompt | Purpose |
|---|---|---|---|---|---|
| **1** | `tool_calling_agent` | `current_time` | Default | none | Minimal working agent |
| **2** | `tool_calling_agent` | `current_time`, Tavily | Default | none | Adds web search via function group |
| **3** | `tool_calling_agent` | `current_time` | 4 | Embedded NASA data | Deterministic single-tool |
| **4** | `tool_calling_agent` | `current_time`, Tavily | 5 | Bounded search rule | Stable multi-tool |
| **5** | `tool_calling_agent` | `current_time`, `python_engine`, Tavily | 5 | 4-step plan | Full pipeline with code execution |

---

## Bottom Line

The notebook progressively demonstrates the **minimum viable configuration**, the **extension mechanism for external tools**, the **discipline required to keep tool-calling agents bounded**, and finally a **full multi-tool pipeline** — all built on one consistent stack: `openai/gpt-oss-20b` + `tool_calling_agent` + NVIDIA NIM. Every case runs successfully. The progression illustrates that most NAT failures in practice come from model choice, agent-type mismatch, and unbounded tool loops — not from the framework itself.